# 04 — Model Architecture Comparison

Train **TensorNet** (MatGL), **CHGNet**, and **ALIGNN** on the same dataset
with identical splits, then compare their regression performance.

| Model | Library | Graph type | Key idea |
|---|---|---|---|
| **TensorNet** | matgl (PyG) | Atom graph | Tensor-product message passing |
| **CHGNet** | chgnet | Atom + bond-angle graph | Charge-informed GNN with 3-body angles |
| **ALIGNN** | alignn (DGL) | Atom graph + line graph | Alternating atom–bond message passing |

**If something fails**

- **`ImportError: libcusparseLt.so.0`** — Common when **torch** is in `~/.local` but **cuSPARSELt** is only under **conda** (`/opt/conda/...`). The first cell searches conda, `CONDA_PREFIX`, and `/opt/conda`, sets `LD_LIBRARY_PATH`, and **preloads** `libcusparseLt.so.0` before importing torch. If it still fails: `pip install nvidia-cusparselt-cu12` (match your CUDA), or use a single environment for torch + NVIDIA libs.
- **`CUDA error: unknown error`** — Often WSL/GPU driver or VRAM; try **`BATCH_SIZE = 16`**, restart the kernel, or set **`ACCELERATOR = "cpu"`** (slow but stable). Long runs can crash mid-epoch; the training loop continues with the next model if one architecture fails.
- **`version_1` incomplete** — A crashed run only wrote partial `metrics.csv`. Re-train or set **`METRICS_VERSION = 0`** to plot an older complete run.

In [1]:
# --- CUDA libs FIRST (before torch) -----------------------------------------
# PyTorch in ~/.local often loads before conda site-packages are on LD_LIBRARY_PATH.
import ctypes
import glob
import importlib.util
import json
import logging
import os
import sys
import sysconfig
import time
import warnings
from pathlib import Path

_pyv = f"python{sys.version_info.major}.{sys.version_info.minor}"


def _collect_nvidia_cusparselt_dirs():
    found, seen = [], set()

    def add(p):
        if p.is_dir():
            s = str(p.resolve())
            if s not in seen:
                seen.add(s)
                found.append(p)

    # This interpreter's real site-packages (most reliable for Jupyter)
    for key in ("platlib", "purelib"):
        try:
            add(Path(sysconfig.get_path(key)) / "nvidia" / "cusparselt" / "lib")
        except Exception:
            pass

    try:
        spec = importlib.util.find_spec("torch")
        if spec and spec.submodule_search_locations:
            site = Path(list(spec.submodule_search_locations)[0]).parent
            add(site / "nvidia" / "cusparselt" / "lib")
            add(site / "nvidia" / "cudnn" / "lib")
    except Exception:
        pass

    for base in (Path(sys.prefix), Path(getattr(sys, "base_prefix", sys.prefix))):
        sp = base / "lib" / _pyv / "site-packages"
        if sp.is_dir():
            add(sp / "nvidia" / "cusparselt" / "lib")

    cp = os.environ.get("CONDA_PREFIX")
    if cp:
        sp = Path(cp) / "lib" / _pyv / "site-packages"
        if sp.is_dir():
            add(sp / "nvidia" / "cusparselt" / "lib")

    try:
        import site as _site
        for sp in map(Path, _site.getsitepackages()):
            add(sp / "nvidia" / "cusparselt" / "lib")
        add(Path(_site.getusersitepackages()) / "nvidia" / "cusparselt" / "lib")
    except Exception:
        pass

    for pattern in (
        str(Path.home() / ".local/lib/python*/site-packages/nvidia/cusparselt/lib"),
        "/opt/conda/lib/python*/site-packages/nvidia/cusparselt/lib",
        "/usr/local/lib/python*/site-packages/nvidia/cusparselt/lib",
    ):
        for d in glob.glob(pattern):
            add(Path(d))

    for guess in (Path("/opt/conda"), Path("/usr/local")):
        sp = guess / "lib" / _pyv / "site-packages"
        if sp.is_dir():
            add(sp / "nvidia" / "cusparselt" / "lib")

    return found


_nv = _collect_nvidia_cusparselt_dirs()
if _nv:
    _ld = ":".join(str(p) for p in _nv)
    os.environ["LD_LIBRARY_PATH"] = _ld + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH ←", _ld[:140] + ("…" if len(_ld) > 140 else ""))
    _ok = False
    for d in _nv:
        so = d / "libcusparseLt.so.0"
        if so.is_file():
            try:
                ctypes.CDLL(str(so), mode=ctypes.RTLD_GLOBAL)
                print("Preloaded", so)
                _ok = True
                break
            except OSError as e:
                print("Preload error:", so, e)
    if not _ok:
        print("WARNING: no libcusparseLt.so.0 in searched dirs; `import torch` may fail.")
else:
    print(
        "WARNING: nvidia/cusparselt/lib not found. Install:  python -m pip install nvidia-cusparselt-cu12"
    )

# Project root
os.chdir(Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd())
if Path("notebooks").is_dir():
    pass
elif Path("../data").is_dir():
    os.chdir("..")
print("Working directory:", Path.cwd())

import numpy as np
import matplotlib.pyplot as plt
import torch
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from sklearn.model_selection import train_test_split
from matgl.ext._pymatgen_pyg import get_element_list

from matprop_nn.models import get_engine, AVAILABLE_ARCHS
from matprop_nn.tasks.train import load_structures_and_targets
from matprop_nn.tasks.regression import RegressionModule

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
print("Available architectures:", AVAILABLE_ARCHS)

Working directory: /workspace
Prepended to LD_LIBRARY_PATH: /home/karasu/.local/lib/python3.11/site-packages/nvidia/cudnn/lib:/opt/conda/lib/python3.11/site-packages/nvidia/cuspars…
Preloaded /opt/conda/lib/python3.11/site-packages/nvidia/cusparselt/lib/libcusparseLt.so.0


/home/karasu/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Available architectures: ['tensornet', 'chgnet', 'alignn']


## 1. Configuration

All hyperparameters live here — nothing is hard-coded below.

In [2]:
# ── Data ────────────────────────────────────────────────────────────
JSON_PATH = "data/mp_dielectric.json"
TARGET_KEY = "e_total"
MAX_SAMPLES = None        # None = use full dataset
LOG_TARGET = True         # log1p-transform skewed targets

# ── Shared training params ──────────────────────────────────────────
MAX_EPOCHS = 100
BATCH_SIZE = 32
LR = 5e-4
SEED = 42
TEST_FRAC = 0.10
VAL_FRAC = 0.10
ACCELERATOR = "gpu"

# ── Per-architecture model config ──────────────────────────────────
ARCH_CONFIGS = {
    "tensornet": dict(
        cutoff=5.0, units=128, nblocks=3, num_rbf=32,
        is_intensive=True, readout_type="weighted_atom",
    ),
    "chgnet": dict(
        pretrained=True,
    ),
    "alignn": dict(
        cutoff=8.0, units=64,
        alignn_layers=4, gcn_layers=4, hidden_features=256,
    ),
}

LOG_DIR = "outputs/comparison"
# Training curves (§6): None = use `log_dir` from the training cell above; int = that Lightning
# subfolder only, e.g. 1 -> outputs/comparison/tensornet/version_1/metrics.csv
METRICS_VERSION = 1

## 2. Shared data loading & splitting

All models see exactly the same structures and the same train / val / test indices.

In [3]:
structures, targets, material_ids = load_structures_and_targets(JSON_PATH, TARGET_KEY)
print(f"Raw samples: {len(structures)}")

targets_np = np.array(targets)

# Remove extreme outliers (keep data within 1st–99th percentile)
p01, p99 = np.percentile(targets_np, 1), np.percentile(targets_np, 99)
mask = (targets_np >= p01) & (targets_np <= p99)
structures = [s for s, m in zip(structures, mask) if m]
targets = [t for t, m in zip(targets, mask) if m]
material_ids = [m for m, ok in zip(material_ids, mask) if ok]
targets_np = np.array(targets)
print(f"After outlier filter (p1–p99 = {p01:.1f}–{p99:.1f}): {len(structures)}")

# Optional subsample for faster iteration
if MAX_SAMPLES is not None and len(structures) > MAX_SAMPLES:
    rng = np.random.RandomState(SEED)
    sel = rng.choice(len(structures), MAX_SAMPLES, replace=False)
    sel.sort()
    structures = [structures[i] for i in sel]
    targets = [targets[i] for i in sel]
    material_ids = [material_ids[i] for i in sel]
    targets_np = np.array(targets)
    print(f"Subsampled to {MAX_SAMPLES}")

if LOG_TARGET:
    targets = [float(np.log1p(t)) for t in targets]
    targets_np = np.array(targets)
    print(f"Applied log1p transform to targets")

data_mean, data_std = float(targets_np.mean()), float(max(targets_np.std(), 1e-6))
print(f"Final: {len(structures)} samples  |  mean={data_mean:.2f}  std={data_std:.2f}")

element_types = get_element_list(structures)
print(f"Element types: {len(element_types)} ({', '.join(element_types[:10])}{'...' if len(element_types) > 10 else ''})")

indices = list(range(len(structures)))
train_val_idx, test_idx = train_test_split(indices, test_size=TEST_FRAC, random_state=SEED)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=VAL_FRAC / (1 - TEST_FRAC), random_state=SEED,
)
print(f"Split → train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}")

INFO | matprop_nn.tasks.train | Loaded 7327 structures with target 'e_total'.


Raw samples: 7327
After outlier filter (p1–p99 = 2.9–180.6): 7179
Applied log1p transform to targets
Final: 7179 samples  |  mean=2.64  std=0.65
Element types: 80 (H, Li, Be, B, C, N, O, F, Na, Mg...)
Split → train 5743 / val 718 / test 718


## 3. Train all architectures

Loop over each architecture, build its dataset / model / dataloaders
through the engine, then train with the shared ``RegressionModule``.

In [ ]:
results = {}  # arch -> {test_MAE, test_RMSE, train_time, n_params}

for arch in ARCH_CONFIGS:
    print(f"\n{'='*60}")
    print(f"  Training: {arch}")
    print(f"{'='*60}")

    engine = get_engine(arch)
    cfg = ARCH_CONFIGS[arch]

    # Datasets (engine-specific graph conversion)
    train_ds, val_ds, test_ds = engine.prepare_datasets(
        structures, targets, train_idx, val_idx, test_idx,
        element_types=element_types,
        cache_dir=f"{LOG_DIR}/cache_{arch}",
        target_key=TARGET_KEY,
        **cfg,
    )

    # Dataloaders
    train_loader, val_loader, test_loader = engine.build_dataloaders(
        train_ds, val_ds, test_ds, batch_size=BATCH_SIZE, num_workers=0,
    )

    # Model
    model = engine.build_model(element_types=element_types, **cfg)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable parameters: {n_params:,}")

    # Lightning module + Trainer
    lit = RegressionModule(model, engine, data_mean=data_mean, data_std=data_std, lr=LR, log_target=LOG_TARGET)
    csv_logger = CSVLogger(LOG_DIR, name=arch)

    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator=ACCELERATOR,
        logger=csv_logger,
        default_root_dir=LOG_DIR,
        enable_progress_bar=True,
    )

    t0 = time.perf_counter()
    trainer.fit(lit, train_dataloaders=train_loader, val_dataloaders=val_loader)
    train_time = time.perf_counter() - t0

    test_out = trainer.test(lit, dataloaders=test_loader, verbose=False)

    results[arch] = {
        "test_MAE":  test_out[0]["test_MAE"],
        "test_RMSE": test_out[0]["test_RMSE"],
        "train_time": train_time,
        "n_params":  n_params,
        "log_dir":   csv_logger.log_dir,
    }
    print(f"  → MAE={results[arch]['test_MAE']:.4f}  "
          f"RMSE={results[arch]['test_RMSE']:.4f}  "
          f"time={train_time:.1f}s")


  Training: tensornet


Processing...
Done!
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


  Trainable parameters: 1,056,002


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ TensorNet         │  1.1 M │ train │     0 │
│ 1 │ loss_fn │ MSELoss           │      0 │ train │     0 │
│ 2 │ mae     │ MeanAbsoluteError │      0 │ train │     0 │
│ 3 │ rmse    │ MeanSquaredError  │      0 │ train │     0 │
└───┴─────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.1 M                                                                                            
Non-trainable params: 32                                                                                           
Total params: 1.1 M                                                                                                
Total estimated model params size (MB): 4                                                                          
Modules in train mode: 73                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

## 4. Summary table

In [ ]:
import pandas as pd

summary = pd.DataFrame(results).T
summary.index.name = "architecture"
summary["n_params"] = summary["n_params"].astype(int)
summary = summary[["n_params", "test_MAE", "test_RMSE", "train_time"]]
summary.columns = ["Parameters", "Test MAE", "Test RMSE", "Train time (s)"]
summary

## 5. Comparison plots

In [ ]:
archs = list(results.keys())
maes  = [results[a]["test_MAE"] for a in archs]
rmses = [results[a]["test_RMSE"] for a in archs]
times = [results[a]["train_time"] for a in archs]
params = [results[a]["n_params"] for a in archs]

colors = {"tensornet": "#5C6BC0", "chgnet": "#EF5350", "alignn": "#66BB6A"}
cs = [colors.get(a, "#888") for a in archs]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# (a) Test MAE
ax = axes[0]
ax.bar(archs, maes, color=cs, edgecolor="k", linewidth=0.5)
ax.set_ylabel("Test MAE")
ax.set_title("(a) Test MAE")
for i, v in enumerate(maes):
    ax.text(i, v + max(maes)*0.02, f"{v:.2f}", ha="center", fontsize=10)

# (b) Parameters
ax = axes[1]
ax.bar(archs, [p / 1000 for p in params], color=cs, edgecolor="k", linewidth=0.5)
ax.set_ylabel("Parameters (K)")
ax.set_title("(b) Model size")
for i, v in enumerate(params):
    ax.text(i, v/1000 + max(params)/1000*0.02, f"{v/1000:.1f}K", ha="center", fontsize=10)

# (c) Training time
ax = axes[2]
ax.bar(archs, times, color=cs, edgecolor="k", linewidth=0.5)
ax.set_ylabel("Time (s)")
ax.set_title("(c) Training wall-time")
for i, v in enumerate(times):
    ax.text(i, v + max(times)*0.02, f"{v:.1f}s", ha="center", fontsize=10)

fig.suptitle(f"Model comparison — {TARGET_KEY} prediction ({MAX_EPOCHS} epochs)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Training curves

Read the CSV logs produced by Lightning to plot loss vs. epoch for each model.

Set `METRICS_VERSION` in the config cell: `1` uses only `outputs/comparison/<arch>/version_1/`; `None` uses the `log_dir` from the training run in this session (so curves match a fresh train).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

_curve_archs = list(ARCH_CONFIGS) if METRICS_VERSION is not None else archs
for arch in _curve_archs:
    if METRICS_VERSION is not None:
        metrics_file = Path(LOG_DIR) / arch / f"version_{METRICS_VERSION}" / "metrics.csv"
    else:
        metrics_file = Path(results[arch]["log_dir"]) / "metrics.csv"
    if not metrics_file.exists():
        print(f"skip {arch}: no {metrics_file}")
        continue
    df = pd.read_csv(metrics_file)

    c = colors.get(arch, "#888")

    # Validation loss per epoch
    if "val_loss" in df.columns:
        val = df.dropna(subset=["val_loss"])
        if not val.empty:
            axes[0].plot(val["epoch"], val["val_loss"], "-o", label=arch, color=c, markersize=4)

    # Validation MAE per epoch
    if "val_MAE" in df.columns:
        val = df.dropna(subset=["val_MAE"])
        if not val.empty:
            axes[1].plot(val["epoch"], val["val_MAE"], "-s", label=arch, color=c, markersize=4)

    # Validation RMSE per epoch (logged with prog_bar=False but still in CSV)
    if "val_RMSE" in df.columns:
        val = df.dropna(subset=["val_RMSE"])
        if not val.empty:
            axes[2].plot(val["epoch"], val["val_RMSE"], "-^", label=arch, color=c, markersize=4)

axes[0].set(xlabel="Epoch", ylabel="Val loss (MSE)", title="(a) Validation loss")
axes[0].legend()
axes[1].set(xlabel="Epoch", ylabel="Val MAE", title="(b) Validation MAE")
axes[1].legend()
axes[2].set(xlabel="Epoch", ylabel="Val RMSE", title="(c) Validation RMSE")
axes[2].legend()
plt.tight_layout()
plt.show()

## 7. Takeaways

After running on your full dataset with more epochs, compare:

1. **Accuracy** — Which architecture achieves the lowest MAE/RMSE?
2. **Efficiency** — Parameters vs. accuracy trade-off.
3. **Speed** — Training wall-time per epoch.
4. **Convergence** — How quickly does each model reach its best validation score?

To switch models for a production run, just change `model.arch` in your
YAML config:

```yaml
model:
  arch: "chgnet"    # or "tensornet" or "alignn"
  cutoff: 5.0
  units: 64
  nblocks: 3
```